# Task 1 — Data Collection & Cleaning

**Goal:** Load the instructor-provided CSV, inspect the data, handle non-informative and missing columns, remove zero-variance features, and save a clean dataset for downstream tasks.

**Input:** `data/raw/chemical_compounds.csv`  
**Output:** `data/processed/compounds_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)

## 1. Load the Data

In [ ]:
df = pd.read_csv('../data/raw/chemical_compounds.csv')
print(f'Shape: {df.shape}')
print(f'Total columns: {len(df.columns)}')
df.head()

In [ ]:
# Check dtypes — identify any non-numeric columns
print('Dtype counts:')
print(df.dtypes.value_counts())
print()
obj_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Non-numeric (object) columns: {obj_cols}')
for c in obj_cols:
    print(f'  {c}: {df[c].nunique()} unique value(s) → {df[c].unique()}')

## 2. Drop Non-Informative Columns

`PUBCHEM_COORDINATE_TYPE` is a string column with a single constant value across all rows — it carries no information and cannot be used as a numeric feature.

In [ ]:
# Drop all non-numeric columns except CID and Class (which are integer IDs/labels)
cols_before = df.shape[1]
df = df.drop(columns=obj_cols)
print(f'Dropped {len(obj_cols)} non-numeric column(s): {obj_cols}')
print(f'Columns: {cols_before} → {df.shape[1]}')

## 3. Inspect Class Labels

In [ ]:
print('Class distribution:')
print(df['Class'].value_counts().sort_index())
print()
print('Class balance (%):')
print(df['Class'].value_counts(normalize=True).sort_index().mul(100).round(1))

In [ ]:
# Sort by class index (0 then 1) so bar order matches the labels below
class_counts = df['Class'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='black')
ax.set_xticklabels(['Inactive (0)', 'Active (1)'], rotation=0)
ax.set_title('Class Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(class_counts):
    ax.text(i, v + 1, str(v), ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../results/figures/class_distribution.png', dpi=150)
plt.show()

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_pct', ascending=False)

print(f'Columns with any missing values: {len(missing_summary)}')
print(missing_summary)

In [ ]:
# Drop columns with more than 20% missing
threshold = 0.20
cols_to_drop = missing_summary[missing_summary['missing_pct'] > threshold * 100].index.tolist()
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f'Dropped {len(cols_to_drop)} column(s) with >{threshold*100:.0f}% missing: {cols_to_drop}')
else:
    print(f'No columns exceed {threshold*100:.0f}% missing — none dropped.')

In [ ]:
# Fill remaining missing values with column median
feature_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in ['CID', 'Class']]
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())
print(f'Missing values remaining: {df.isnull().sum().sum()}')

## 5. Remove Zero-Variance Features

Columns with zero variance have the same value for every compound — they carry no information for classification.

In [ ]:
X = df[feature_cols]
selector = VarianceThreshold(threshold=0)
selector.fit(X)

zero_var_cols = [feature_cols[i] for i, v in enumerate(selector.variances_) if v == 0]
print(f'Zero-variance columns to drop: {len(zero_var_cols)}')
print(zero_var_cols)

In [ ]:
df = df.drop(columns=zero_var_cols)
# Recompute feature list after dropping zero-variance columns
feature_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in ['CID', 'Class']]
print(f'Features remaining: {len(feature_cols)}')

## 6. Final Summary

In [ ]:
print(f'Final shape: {df.shape}')
print(f'  Compounds: {df.shape[0]}')
print(f'  Features:  {len(feature_cols)}')
print(f'  Metadata:  CID, Class')
print()
print('Class distribution (final):')
print(df['Class'].value_counts().sort_index())

## 7. Save Cleaned Data

In [ ]:
df.to_csv('../data/processed/compounds_clean.csv', index=False)
print('Saved to data/processed/compounds_clean.csv')